# HEAL-CITY — Feature Engineering

This notebook documents the feature engineering phase of the HEAL-CITY smart city healthcare dataset. We transform raw and cleaned data into scalable, comparable ratios, extract detailed disease summaries, handle accessibility/mismatch placeholders, and normalize variables for scoring.

## 01. Load Master Dataset
First, we import core libraries and load `master_heal_city.csv`.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

master_df = pd.read_csv("../dataset/processed/master_heal_city.csv")
print("Master dataset dimensions:", master_df.shape)
display(master_df.head(5))

## 02. Load Supporting Data
Load the detailed cleaned disease table `clean_penyakit.csv` to construct specific disease metrics.

In [ ]:
penyakit_df = pd.read_csv("../dataset/cleaned/clean_penyakit.csv")
print("Penyakit dataset dimensions:", penyakit_df.shape)
display(penyakit_df.head(5))

## 03. Population Features
Ensure population values are standard and represent potential demand size.

In [ ]:
df_feat = master_df[["kecamatan", "jumlah_penduduk", "kepadatan_penduduk"]].copy()
display(df_feat.head(5))

## 04. Demand Features
Calculate normalized service pressure (`visits_per_1000`) based on visit volume.

In [ ]:
df_feat["total_kunjungan"] = master_df["total_kunjungan"]
df_feat["visits_per_1000"] = df_feat["total_kunjungan"] / df_feat["jumlah_penduduk"]
display(df_feat[["kecamatan", "total_kunjungan", "visits_per_1000"]].head(5))

## 05. Workforce Features
Compute workforce capacities (total nakes, nurse, and midwife ratios per 1,000 population).

In [ ]:
df_feat["total_tenaga_kesehatan"] = master_df["total_tenaga_kesehatan"]
df_feat["jumlah_perawat"] = master_df["jumlah_perawat"]
df_feat["jumlah_bidan"] = master_df["jumlah_bidan"]

df_feat["nakes_per_1000"] = df_feat["total_tenaga_kesehatan"] / df_feat["jumlah_penduduk"]
df_feat["perawat_per_1000"] = df_feat["jumlah_perawat"] / df_feat["jumlah_penduduk"]
df_feat["bidan_per_1000"] = df_feat["jumlah_bidan"] / df_feat["jumlah_penduduk"]

display(df_feat[["kecamatan", "nakes_per_1000", "perawat_per_1000", "bidan_per_1000"]].head(5))

## 06. Facility Features
Compute density rates for total faskes, primary care puskesmas, and auxiliary pustu per 100,000 population.

In [ ]:
df_feat["total_faskes"] = master_df["total_faskes"]
df_feat["jumlah_puskesmas"] = master_df["jumlah_puskesmas"]
df_feat["jumlah_pustu"] = master_df["jumlah_pustu"]

df_feat["faskes_per_100k"] = (df_feat["total_faskes"] / df_feat["jumlah_penduduk"]) * 100.0
df_feat["puskesmas_per_100k"] = (df_feat["jumlah_puskesmas"] / df_feat["jumlah_penduduk"]) * 100.0
df_feat["pustu_per_100k"] = (df_feat["jumlah_pustu"] / df_feat["jumlah_penduduk"]) * 100.0

display(df_feat[["kecamatan", "faskes_per_100k", "puskesmas_per_100k"]].head(5))

## 07. Bed Features
Examine inpatient bed capacities per 1,000 population.

In [ ]:
df_feat["total_tempat_tidur"] = master_df["total_tempat_tidur"]
df_feat["beds_per_1000"] = df_feat["total_tempat_tidur"] / df_feat["jumlah_penduduk"]
display(df_feat[["kecamatan", "total_tempat_tidur", "beds_per_1000"]].head(5))

## 08. Disease Features
Calculate overall disease burden per 1,000 residents and extract disease-specific dominancy metrics.

In [ ]:
df_feat["total_kasus_penyakit"] = master_df["total_kasus_penyakit"]
df_feat["disease_per_1000"] = df_feat["total_kasus_penyakit"] / df_feat["jumlah_penduduk"]

# Group to find highest disease caseload and count
df_peny_grouped = penyakit_df.groupby(["kecamatan", "jenis_penyakit"])["jumlah_kasus"].sum().reset_index()
disease_spec_records = []
for kec in df_feat["kecamatan"]:
    kec_df = df_peny_grouped[df_peny_grouped["kecamatan"] == kec]
    unique_diseases_count = kec_df[kec_df["jumlah_kasus"] > 0]["jenis_penyakit"].nunique()
    active_cases = kec_df[kec_df["jumlah_kasus"] > 0]
    if len(active_cases) > 0:
        max_row = active_cases.loc[active_cases["jumlah_kasus"].idxmax()]
        dominant_disease = max_row["jenis_penyakit"]
        highest_case_count = max_row["jumlah_kasus"]
    else:
        dominant_disease = pd.NA
        highest_case_count = 0
    disease_spec_records.append({
        "kecamatan": kec,
        "jenis_penyakit_dominan": dominant_disease,
        "kasus_penyakit_tertinggi": highest_case_count,
        "jumlah_jenis_penyakit": unique_diseases_count
    })

df_dis_spec = pd.DataFrame(disease_spec_records)
df_feat = pd.merge(df_feat, df_dis_spec, on="kecamatan", how="left")
display(df_feat[["kecamatan", "disease_per_1000", "jenis_penyakit_dominan", "jumlah_jenis_penyakit"]].head(5))

## 09. Workforce-Demand Mismatch
As we operate with single-year records, historical cross-year growth rates are not calculable. Growth and Mismatch features are designated as `NaN` placeholders.

In [ ]:
df_feat["workforce_growth"] = np.nan
df_feat["demand_growth"] = np.nan
df_feat["workforce_demand_mismatch"] = np.nan
print("Growth mismatch placeholders created as NaN.")

## 10. Accessibility Features
Due to lack of spatial GIS coordinates in this iteration, travel distances and times are set as `NaN` placeholders.

In [ ]:
df_feat["distance_to_facility"] = np.nan
df_feat["travel_time"] = np.nan
df_feat["accessibility_gap"] = np.nan
print("Accessibility placeholders initialized.")

## 11. Normalization
Define min-max scaling function to project features into comparable range `[0, 1]`.

In [ ]:
def min_max_norm(series):
    if series.max() == series.min():
        return series * 0.0
    return (series - series.min()) / (series.max() - series.min())

print("Min-max scaling utility declared.")

## 12. Directionality
Scale the scores such that a higher value represents a worse condition (higher gap):

In [ ]:
# Demand (higher is worse)
df_feat["population_demand_score"] = min_max_norm(df_feat["jumlah_penduduk"])
df_feat["visits_pressure_score"] = min_max_norm(df_feat["visits_per_1000"])
df_feat["disease_need_score"] = min_max_norm(df_feat["disease_per_1000"])

# Capacity (higher is better -> Gap = 1 - Norm)
df_feat["workforce_gap"] = 1.0 - min_max_norm(df_feat["nakes_per_1000"])
df_feat["facility_gap"] = 1.0 - min_max_norm(df_feat["faskes_per_100k"])
df_feat["bed_gap"] = 1.0 - min_max_norm(df_feat["beds_per_1000"])

display(df_feat[["kecamatan", "population_demand_score", "workforce_gap", "facility_gap"]].head(5))

## 13. Correlation Check
Verify correlation coefficients among the newly computed features to prevent double-counting redundancy.

In [ ]:
feat_cols = [
    "population_demand_score", "visits_pressure_score", "disease_need_score",
    "workforce_gap", "facility_gap", "bed_gap"
]
corr = df_feat[feat_cols].corr()
fig = px.imshow(corr, text_auto=True, title="Feature Correlation Matrix")
fig.show()

## 14. Feature Validation
Ensure that zero populations or denominators did not generate negative, missing, or infinite values in our ratio columns.

In [ ]:
numeric_cols = df_feat.select_dtypes(include="number").columns
inf_sums = np.isinf(df_feat[numeric_cols]).sum()
nan_sums = df_feat[numeric_cols].isna().sum()
print("Infinite value counts per column:")
print(inf_sums[inf_sums > 0])
print("\nNaN value counts per column:")
print(nan_sums[nan_sums > 0])

## 15. Feature Dictionary
Display feature definition dictionary log.

In [ ]:
dict_df = pd.read_csv("../logs/feature_dictionary.csv")
display(dict_df)

## 16. Export Feature Dataset
Verify exported CSV format.

In [ ]:
feat_df_loaded = pd.read_csv("../dataset/processed/heal_city_features.csv")
print("Exported Features Dimensions:", feat_df_loaded.shape)
display(feat_df_loaded.head(5))